In [67]:
articles = {
    "biology": [
        "List_of_parasites_of_the_marsh_rice_rat",
        "Marsh_rice_rat",
        "Oryzomys_albiventer",
        "Oryzomys_antillarum",
        "Oryzomys",
        "Oryzomys_couesi",
        "Oryzomys_dimidiatus",
        "Oryzomys_gorgasi",
        "Oryzomys_nelsoni",
        "Oryzomys_peninsulae"
    ],
    
    "physics": [
        "Ed_White_(astronaut)",
        "Elliot_See",
        "Frank_Borman",
        "James_McDivitt",
        "Jim_Lovell",
        "John_Young_(astronaut)",
        "NASA_Astronaut_Group_2",
        "Neil_Armstrong",
        "Pete_Conrad",
        "Thomas_P._Stafford"
    ],

    "religion": [
        "Augustine_of_Canterbury",
        "Honorius_of_Canterbury",
        "James_the_Deacon",
        "Justus",
        "Laurence_of_Canterbury",
        "List_of_members_of_the_Gregorian_mission",
        "Mellitus",
        "Paulinus_of_York",
        "Peter_of_Canterbury",
        "Romanus_(bishop_of_Rochester)"
    ]
}

In [68]:
import wikipediaapi

def gather_article_texts(articles: dict[str, list[str]]) -> dict[str, list[tuple[str, str]]]:
    wiki_wiki = wikipediaapi.Wikipedia(
        user_agent='MyProjectName (merlin@example.com)',
        language='en',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    return {
        topic: [
            (title, wiki_wiki.page(title).text) for title in titles
        ] 
        for topic, titles in articles.items()
    }

In [69]:
article_texts = gather_article_texts(articles)

In [100]:
for topic, pages in article_texts.items():
    print(f"topic = {topic}")
    for title, text in pages:
        print(f"\t{title:<30}: {text[:50]}")

topic = biology
	List_of_parasites_of_the_marsh_rice_rat: A variety of parasites have been recorded from the
	Marsh_rice_rat                : The marsh rice rat (Oryzomys palustris) is a semia
	Oryzomys_albiventer           : Oryzomys albiventer, commonly known as the White-b
	Oryzomys_antillarum           : Oryzomys antillarum, also known as the Jamaican ri
	Oryzomys                      : Oryzomys is a genus of semiaquatic rodents in the 
	Oryzomys_couesi               : Oryzomys couesi, also known as Coues's rice rat, i
	Oryzomys_dimidiatus           : Oryzomys dimidiatus, also known as the Nicaraguan 
	Oryzomys_gorgasi              : Oryzomys gorgasi, also known as Gorgas's oryzomys 
	Oryzomys_nelsoni              : Oryzomys nelsoni, also known as the Nelson’s rice 
	Oryzomys_peninsulae           : Oryzomys peninsulae, also known as the Lower Calif
topic = physics
	Ed_White_(astronaut)          : Edward Higgins White II (November 14, 1930 – Janua
	Elliot_See                    : El

In [101]:
# import re
import nltk

from nltk import word_tokenize, pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)

# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")
# nltk.download("wordnet")
# nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

# stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV

    return wordnet.NOUN


def lemmatize_text(text: str) -> str:
    # Transformare în litere mici
    text = text.lower()

    # Tokenizare
    tokens = word_tokenize(text)

    # Eliminare punctuație și stop words
    tokens = [
        token
        for token in tokens
        if token.isalpha() #and token not in stop_words
    ]

    # POS tagging
    tagged_tokens = pos_tag(tokens)

    # Lematizare
    lemmas = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged_tokens
    ]

    return " ".join(lemmas)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Andier13\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [102]:
lemmatized_article_texts = {
    topic: [
        (title, lemmatize_text(text)) for title, text in pages
    ] 
    for topic, pages in article_texts.items()
}

In [103]:
for topic, pages in lemmatized_article_texts.items():
    print(f"topic = {topic}")
    for title, text in pages:
        print(f"\t{title:<30}: {text[:50]}")

topic = biology
	List_of_parasites_of_the_marsh_rice_rat: a variety of parasite have be record from the mars
	Marsh_rice_rat                : the marsh rice rat oryzomys palustris be a semiaqu
	Oryzomys_albiventer           : oryzomys albiventer commonly know a the rice rat b
	Oryzomys_antillarum           : oryzomys antillarum also know a the jamaican rice 
	Oryzomys                      : oryzomys be a genus of semiaquatic rodent in the t
	Oryzomys_couesi               : oryzomys couesi also know a coue rice rat be a sem
	Oryzomys_dimidiatus           : oryzomys dimidiatus also know a the nicaraguan ory
	Oryzomys_gorgasi              : oryzomys gorgasi also know a gorgas oryzomys or go
	Oryzomys_nelsoni              : oryzomys nelsoni also know a the nelson s rice rat
	Oryzomys_peninsulae           : oryzomys peninsulae also know a the low california
topic = physics
	Ed_White_(astronaut)          : edward higgins white ii november january be an ame
	Elliot_See                    : el

In [104]:
import numpy as np

In [105]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tf_vectorize(lemmatized_article_texts: dict[str, list[tuple[str, str]]]) -> tuple[dict[str, list[tuple[str, np.ndarray]]], TfidfVectorizer]:
    lemmatized_documents = [
        text
        for _, pages in lemmatized_article_texts.items()
        for _, text in pages
    ]
    
    tf_idf_vectorizer = TfidfVectorizer(
        input="content",
        stop_words="english",
        lowercase=True
    )

    tf_idf_vectorizer.fit(lemmatized_documents)

    tf_idf_vectorized_articles = {
        topic: [
            (title, tf_idf_vectorizer.transform([text])[0]) for title, text in pages
        ] 
        for topic, pages in lemmatized_article_texts.items()
    }

    return tf_idf_vectorized_articles, tf_idf_vectorizer

In [106]:
tf_idf_vectorized_articles, tf_idf_vectorizer = tf_vectorize(lemmatized_article_texts)

In [107]:
print("\nTermenii identificați:")
print(tf_idf_vectorizer.get_feature_names_out()[:100])

print("\nVectorizarea TF-IDF a primului articol:")
print(tf_idf_vectorized_articles["biology"][0][1])


Termenii identificați:
['aa' 'aaron' 'abandon' 'abandonment' 'abbess' 'abbey' 'abbot'
 'abbreviate' 'abc' 'abdominal' 'ability' 'abiventer' 'abject' 'able'
 'abnormal' 'aboard' 'abort' 'abre' 'absence' 'absent' 'absorb' 'abstract'
 'abundance' 'abuse' 'abuzeineh' 'academic' 'academy' 'acarology' 'acarus'
 'accept' 'acceptable' 'acceptance' 'accessorium' 'accessory' 'accident'
 'accidentally' 'accommodate' 'accompanied' 'accompany' 'accomplish'
 'accomplished' 'accomplishment' 'accord' 'accordingly' 'account' 'accrue'
 'accumulate' 'accuse' 'achieve' 'achievement' 'acknowledge' 'acoustic'
 'acquire' 'acquisition' 'acre' 'acres' 'acrocentric' 'act' 'acta'
 'acting' 'action' 'activate' 'active' 'actively' 'activity' 'actor'
 'actress' 'actual' 'actually' 'actuate' 'ad' 'adair' 'adapt' 'adaptation'
 'adapter' 'adaptive' 'adapts' 'add' 'addition' 'additional'
 'additionally' 'address' 'addressee' 'adeodatus' 'adept' 'adeptness'
 'adequate' 'adequately' 'adjacent' 'adjust' 'adler' 'administ

In [108]:
from sklearn.feature_extraction.text import CountVectorizer

def tf_vectorize(lemmatized_article_texts: dict[str, list[tuple[str, str]]]) -> tuple[dict[str, list[tuple[str, np.ndarray]]], CountVectorizer]:
    lemmatized_documents = [
        text
        for _, pages in lemmatized_article_texts.items()
        for _, text in pages
    ]
    
    tf_vectorizer = CountVectorizer(
        input="content",
        stop_words="english",
        lowercase=True
    )

    tf_vectorizer.fit(lemmatized_documents)

    tf_vectorized_articles = {
        topic: [
            (title, tf_vectorizer.transform([text])[0]) for title, text in pages
        ] 
        for topic, pages in lemmatized_article_texts.items()
    }

    return tf_vectorized_articles, tf_vectorizer

In [109]:
tf_vectorized_articles, tf_vectorizer = tf_vectorize(lemmatized_article_texts)

In [110]:
print("\nTermenii identificați:")
print(tf_vectorizer.get_feature_names_out()[:100])

print("\nVectorizarea TF a primului articol:")
print(tf_vectorized_articles["biology"][0][1])


Termenii identificați:
['aa' 'aaron' 'abandon' 'abandonment' 'abbess' 'abbey' 'abbot'
 'abbreviate' 'abc' 'abdominal' 'ability' 'abiventer' 'abject' 'able'
 'abnormal' 'aboard' 'abort' 'abre' 'absence' 'absent' 'absorb' 'abstract'
 'abundance' 'abuse' 'abuzeineh' 'academic' 'academy' 'acarology' 'acarus'
 'accept' 'acceptable' 'acceptance' 'accessorium' 'accessory' 'accident'
 'accidentally' 'accommodate' 'accompanied' 'accompany' 'accomplish'
 'accomplished' 'accomplishment' 'accord' 'accordingly' 'account' 'accrue'
 'accumulate' 'accuse' 'achieve' 'achievement' 'acknowledge' 'acoustic'
 'acquire' 'acquisition' 'acre' 'acres' 'acrocentric' 'act' 'acta'
 'acting' 'action' 'activate' 'active' 'actively' 'activity' 'actor'
 'actress' 'actual' 'actually' 'actuate' 'ad' 'adair' 'adapt' 'adaptation'
 'adapter' 'adaptive' 'adapts' 'add' 'addition' 'additional'
 'additionally' 'address' 'addressee' 'adeodatus' 'adept' 'adeptness'
 'adequate' 'adequately' 'adjacent' 'adjust' 'adler' 'administ